# Tutorial: Advanced Problems with Solutions — Decorators That Prime Coroutines

This notebook is a second, independent tutorial on generator-based coroutines.

Instead of presenting large finished programs immediately, every problem is broken
into small logical steps:

1. understand the behavior we want;
2. identify the generator protocol involved;
3. implement one small piece;
4. inspect the state of the coroutine;
5. test normal and abnormal cases;
6. combine the pieces into a larger design.

The examples use only the Python standard library.

> Generator coroutines are different from `async def` coroutines. They are useful
> for understanding cooperative control flow, streaming processors, state machines,
> pipelines, and protocol-oriented design.

## Learning objectives

By completing the notebook, you will practice:

- priming a generator safely with a decorator;
- preserving metadata with `functools.wraps`;
- forwarding arbitrary arguments;
- inspecting `GEN_CREATED`, `GEN_SUSPENDED`, and `GEN_CLOSED`;
- designing request/response protocols around `.send()`;
- injecting control flow with `.throw()`;
- returning a final value through `StopIteration.value`;
- propagating `.close()` through a pipeline;
- building transactional and dynamically routed coroutine systems;
- testing coroutine lifecycle behavior systematically.

## A small mental model

A generator coroutine moves through a sequence of states.

- **Created:** the function was called, but execution has not reached the first `yield`.
- **Suspended:** execution is paused at a `yield`; `.send(value)` can now deliver data.
- **Closed:** execution finished, an unhandled exception escaped, or `.close()` completed.

The first call must be `next(generator)` or `generator.send(None)`.
Sending any other value to a newly created generator raises `TypeError`.

In [1]:
from __future__ import annotations

import inspect
import math
from collections import deque
from collections.abc import Callable, Generator, Iterable
from contextlib import AbstractContextManager
from dataclasses import dataclass
from functools import wraps
from numbers import Real
from typing import Any, Generic, ParamSpec, TypeVar, cast

P = ParamSpec("P")
Y = TypeVar("Y")
S = TypeVar("S")
R = TypeVar("R")

## Demonstration: why priming is required

The following coroutine waits for a value. Before the first `yield` is reached, it
cannot receive a non-`None` value.

In [2]:
def raw_receiver(log: list[str]) -> Generator[None, str, None]:
    while True:
        message = yield
        log.append(message)


raw_log: list[str] = []
raw = raw_receiver(raw_log)

assert inspect.getgeneratorstate(raw) == "GEN_CREATED"

try:
    raw.send("too early")
except TypeError as exc:
    print(type(exc).__name__, "-", exc)

assert inspect.getgeneratorstate(raw) == "GEN_CREATED"

TypeError - can't send non-None value to a just-started generator


Calling `next(raw)` advances execution to the first `yield`. The generator is now
suspended and ready to receive data.

In [3]:
next(raw)

assert inspect.getgeneratorstate(raw) == "GEN_SUSPENDED"

raw.send("accepted")
raw.send("also accepted")

assert raw_log == ["accepted", "also accepted"]

raw.close()
assert inspect.getgeneratorstate(raw) == "GEN_CLOSED"

# Problem 1 — Build a careful auto-priming decorator

## Goal

Create a decorator named `auto_prime` that:

- calls the generator function;
- verifies that the returned object is actually a generator;
- advances it exactly once;
- reports premature termination clearly;
- supports positional and keyword arguments;
- preserves metadata.

We will build it in three steps.

## Step 1: define a domain-specific exception

A dedicated exception makes errors from the priming layer easy to distinguish from
errors raised later by the coroutine's own business logic.

In [4]:
class CoroutineSetupError(RuntimeError):
    """Raised when a decorated coroutine cannot be prepared for use."""

## Step 2: implement the wrapper

The decorator receives a generator function and returns a normal function.
That returned function creates and primes a generator every time it is called.

Notice that the wrapper catches `StopIteration` only during priming. A generator
that stops before its first `yield` does not satisfy the coroutine contract.

In [5]:
def auto_prime(
    generator_function: Callable[P, Generator[Y, S, R]],
) -> Callable[P, Generator[Y, S, R]]:
    """Return a callable that creates an already-primed generator coroutine."""

    @wraps(generator_function)
    def create_and_prime(
        *args: P.args,
        **kwargs: P.kwargs,
    ) -> Generator[Y, S, R]:
        candidate = generator_function(*args, **kwargs)

        if not inspect.isgenerator(candidate):
            raise CoroutineSetupError(
                f"{generator_function.__qualname__} did not return a generator"
            )

        try:
            next(candidate)
        except StopIteration as exc:
            raise CoroutineSetupError(
                f"{generator_function.__qualname__} stopped before its first yield"
            ) from exc

        return cast(Generator[Y, S, R], candidate)

    return create_and_prime

## Step 3: verify metadata, arguments, and state

This example has both positional and keyword-only parameters. Its first yielded
value is `None`, which is consumed by the decorator during priming.

In [6]:
@auto_prime
def tagged_collector(
    target: list[str],
    tag: str,
    *,
    uppercase: bool = False,
) -> Generator[None, str, None]:
    """Collect tagged strings in a caller-provided list."""
    while True:
        value = yield
        if uppercase:
            value = value.upper()
        target.append(f"{tag}:{value}")


collected: list[str] = []
collector = tagged_collector(collected, "EVENT", uppercase=True)

assert tagged_collector.__name__ == "tagged_collector"
assert tagged_collector.__doc__ == "Collect tagged strings in a caller-provided list."
assert inspect.getgeneratorstate(collector) == "GEN_SUSPENDED"

collector.send("started")
collector.send("finished")
collector.close()

assert collected == ["EVENT:STARTED", "EVENT:FINISHED"]

## Edge-case checks

A decorator is part of an API boundary, so its failures should be intentional and
readable. The next two examples test incorrect decorated callables.

In [7]:
@auto_prime
def returns_plain_value() -> Any:
    return "not a generator"


try:
    returns_plain_value()
except CoroutineSetupError as exc:
    assert "did not return a generator" in str(exc)
else:
    raise AssertionError("CoroutineSetupError was expected")

In [8]:
@auto_prime
def stops_immediately() -> Generator[None, None, None]:
    if False:
        yield
    return


try:
    stops_immediately()
except CoroutineSetupError as exc:
    assert "stopped before its first yield" in str(exc)
else:
    raise AssertionError("CoroutineSetupError was expected")

# Problem 2 — Capture the initial yielded value

## Why this is more advanced

A normal priming decorator discards the value produced by the first `yield`.
Sometimes that value is meaningful:

- a readiness message;
- a protocol version;
- a configuration summary;
- an initial state snapshot.

We will create a small wrapper object that stores both the generator and its initial
yielded value.

## Step 1: define the wrapper type

The wrapper forwards `.send()`, `.throw()`, and `.close()` to the underlying
generator while exposing `initial`.

In [9]:
InitialT = TypeVar("InitialT")
InputT = TypeVar("InputT")
ReturnT = TypeVar("ReturnT")


@dataclass
class PrimedEndpoint(Generic[InitialT, InputT, ReturnT]):
    generator: Generator[InitialT, InputT, ReturnT]
    initial: InitialT

    @property
    def state(self) -> str:
        return inspect.getgeneratorstate(self.generator)

    def send(self, value: InputT) -> InitialT:
        return self.generator.send(value)

    def throw(self, error: BaseException | type[BaseException]) -> InitialT:
        return self.generator.throw(error)

    def close(self) -> None:
        self.generator.close()

## Step 2: create a decorator that captures the first yield

This decorator is deliberately separate from `auto_prime`. It returns a richer
object and therefore has a different public interface.

In [10]:
def prime_and_capture(
    generator_function: Callable[P, Generator[InitialT, InputT, ReturnT]],
) -> Callable[P, PrimedEndpoint[InitialT, InputT, ReturnT]]:
    @wraps(generator_function)
    def prepare(
        *args: P.args,
        **kwargs: P.kwargs,
    ) -> PrimedEndpoint[InitialT, InputT, ReturnT]:
        generator = generator_function(*args, **kwargs)

        if not inspect.isgenerator(generator):
            raise CoroutineSetupError(
                f"{generator_function.__qualname__} did not return a generator"
            )

        try:
            initial = next(generator)
        except StopIteration as exc:
            raise CoroutineSetupError(
                f"{generator_function.__qualname__} stopped before initialization"
            ) from exc

        return PrimedEndpoint(generator=generator, initial=initial)

    return prepare

## Step 3: use the initial value as a handshake

The endpoint announces its version before accepting commands.
Every later `.send()` returns a response string.

In [11]:
@prime_and_capture
def versioned_endpoint(
    version: str,
) -> Generator[str, str, None]:
    request = yield f"READY/{version}"

    while True:
        response = f"{version}:{request.strip().upper()}"
        request = yield response


endpoint = versioned_endpoint("2.1")

assert endpoint.initial == "READY/2.1"
assert endpoint.state == "GEN_SUSPENDED"
assert endpoint.send(" ping ") == "2.1:PING"
assert endpoint.send("status") == "2.1:STATUS"

endpoint.close()
assert endpoint.state == "GEN_CLOSED"

# Problem 3 — Design a request/response statistics coroutine

## Goal

Build a coroutine that receives real numbers and returns an immutable snapshot
containing:

- number of accepted values;
- running total;
- mean;
- smallest value;
- largest value.

Invalid values should not close the coroutine. Instead, the previous snapshot
should be returned unchanged.

We will use constant memory.

## Step 1: model the response

A frozen dataclass is useful because callers cannot accidentally mutate the
coroutine's internal state through the returned object.

In [12]:
@dataclass(frozen=True)
class RunningSummary:
    count: int
    total: float
    mean: float
    minimum: float
    maximum: float

## Step 2: decide what the first yield should be

Before any values are accepted, there is no meaningful minimum, maximum, or mean.
Therefore, the initial response is `None`.

After every `.send(value)`, execution updates the state and reaches the next
`yield`, returning the new snapshot.

In [13]:
@auto_prime
def summarize_numbers() -> Generator[RunningSummary | None, Any, None]:
    count = 0
    total = 0.0
    minimum = math.inf
    maximum = -math.inf
    summary: RunningSummary | None = None

    while True:
        value = yield summary

        if isinstance(value, bool) or not isinstance(value, Real):
            continue

        numeric = float(value)

        if not math.isfinite(numeric):
            continue

        count += 1
        total += numeric
        minimum = min(minimum, numeric)
        maximum = max(maximum, numeric)

        summary = RunningSummary(
            count=count,
            total=total,
            mean=total / count,
            minimum=minimum,
            maximum=maximum,
        )

## Step 3: test accepted values

The return value of `.send()` is the value yielded after processing the sent input.

In [14]:
summary_coroutine = summarize_numbers()

first = summary_coroutine.send(10)
second = summary_coroutine.send(20)
third = summary_coroutine.send(-6)

assert first == RunningSummary(1, 10.0, 10.0, 10.0, 10.0)
assert second == RunningSummary(2, 30.0, 15.0, 10.0, 20.0)
assert third == RunningSummary(3, 24.0, 8.0, -6.0, 20.0)

## Step 4: test rejected values

A boolean is rejected explicitly because `bool` is a subclass of `int`.
Non-finite floating-point values are also ignored.

In [15]:
unchanged_1 = summary_coroutine.send("not numeric")
unchanged_2 = summary_coroutine.send(True)
unchanged_3 = summary_coroutine.send(float("nan"))
unchanged_4 = summary_coroutine.send(float("inf"))

assert unchanged_1 == third
assert unchanged_2 == third
assert unchanged_3 == third
assert unchanged_4 == third
assert inspect.getgeneratorstate(summary_coroutine) == "GEN_SUSPENDED"

summary_coroutine.close()

# Problem 4 — Control a coroutine with injected exceptions

## Goal

Use `.throw()` to inject control signals into a running coroutine.

The coroutine will maintain a numeric total and support:

- normal numbers through `.send(number)`;
- `Freeze` through `.throw(Freeze)`;
- `Unfreeze` through `.throw(Unfreeze)`;
- `Clear` through `.throw(Clear)`.

While frozen, sent numbers are ignored.

## Step 1: define narrow protocol exceptions

These are not error conditions. They are control messages represented as exception
types because `.throw()` raises them at the suspended `yield`.

In [16]:
class Freeze(Exception):
    pass


class Unfreeze(Exception):
    pass


class Clear(Exception):
    pass

## Step 2: model the state returned to the caller

In [17]:
@dataclass(frozen=True)
class TotalState:
    total: float
    frozen: bool

## Step 3: implement the control loop

The `try` statement surrounds the `yield` because injected exceptions are raised
at that exact suspension point.

In [18]:
@auto_prime
def controlled_total() -> Generator[TotalState, Any, None]:
    total = 0.0
    frozen = False
    state = TotalState(total=total, frozen=frozen)

    while True:
        try:
            value = yield state
        except Freeze:
            frozen = True
        except Unfreeze:
            frozen = False
        except Clear:
            total = 0.0
        else:
            if (
                not frozen
                and not isinstance(value, bool)
                and isinstance(value, Real)
            ):
                total += float(value)

        state = TotalState(total=total, frozen=frozen)

## Step 4: exercise both data and control paths

In [19]:
total = controlled_total()

assert total.send(5) == TotalState(5.0, False)
assert total.send(2.5) == TotalState(7.5, False)

assert total.throw(Freeze) == TotalState(7.5, True)
assert total.send(100) == TotalState(7.5, True)

assert total.throw(Unfreeze) == TotalState(7.5, False)
assert total.send(1) == TotalState(8.5, False)

assert total.throw(Clear) == TotalState(0.0, False)

total.close()

## Important failure mode

An exception that is not caught by the coroutine escapes and closes it.
This is why control protocols should catch only their own expected exception types.

In [20]:
unprotected = controlled_total()

try:
    unprotected.throw(KeyError("unexpected"))
except KeyError:
    pass

assert inspect.getgeneratorstate(unprotected) == "GEN_CLOSED"

# Problem 5 — Build a tutorial pipeline one stage at a time

## Pipeline objective

Process raw strings using four stages:

1. trim whitespace;
2. keep only non-empty strings;
3. convert text to uppercase;
4. collect results in a list.

Each stage sends its output to the next stage.
Closing the first stage should close every downstream stage.

## Step 1: create the terminal sink

A sink consumes values but does not produce a meaningful response.

In [21]:
@auto_prime
def list_sink(target: list[str]) -> Generator[None, str, None]:
    while True:
        target.append((yield))

## Step 2: create a generic mapping stage

The mapping stage applies one function and forwards the result.
Cleanup belongs in `finally`, ensuring that downstream stages are closed even when
the upstream caller exits because of an error.

In [22]:
@auto_prime
def mapping_stage(
    transform: Callable[[Any], Any],
    target: Generator[Any, Any, Any],
) -> Generator[None, Any, None]:
    try:
        while True:
            item = yield
            target.send(transform(item))
    finally:
        target.close()

## Step 3: create a generic filtering stage

In [23]:
@auto_prime
def filtering_stage(
    predicate: Callable[[Any], bool],
    target: Generator[Any, Any, Any],
) -> Generator[None, Any, None]:
    try:
        while True:
            item = yield
            if predicate(item):
                target.send(item)
    finally:
        target.close()

## Step 4: assemble from the sink backward

A pipeline is usually easiest to construct from the terminal stage toward the
entry stage.

In [24]:
pipeline_results: list[str] = []

sink_stage = list_sink(pipeline_results)
uppercase_stage = mapping_stage(str.upper, sink_stage)
non_empty_stage = filtering_stage(lambda text: len(text) > 0, uppercase_stage)
entry_stage = mapping_stage(str.strip, non_empty_stage)

## Step 5: send data only to the entry stage

In [25]:
for raw_text in ["  alpha ", "", "   ", "Beta", " gamma"]:
    entry_stage.send(raw_text)

assert pipeline_results == ["ALPHA", "BETA", "GAMMA"]

## Step 6: verify closure propagation

Only the first stage is closed explicitly. Every stage should end in
`GEN_CLOSED`.

In [26]:
entry_stage.close()

assert inspect.getgeneratorstate(entry_stage) == "GEN_CLOSED"
assert inspect.getgeneratorstate(non_empty_stage) == "GEN_CLOSED"
assert inspect.getgeneratorstate(uppercase_stage) == "GEN_CLOSED"
assert inspect.getgeneratorstate(sink_stage) == "GEN_CLOSED"

# Problem 6 — Add batching to a pipeline

## Goal

Create a stage that groups incoming values into fixed-size tuples.

For example, with a batch size of three:

```text
1, 2, 3, 4, 5, 6, 7
```

should produce:

```text
(1, 2, 3), (4, 5, 6)
```

By default, the incomplete final batch is discarded when the coroutine closes.
An optional flag will allow the final partial batch to be flushed.

## Step 1: validate configuration before the first yield

A key detail: because the decorator primes immediately, validation performed before
the first `yield` happens during construction.

In [27]:
@auto_prime
def batch_stage(
    size: int,
    target: Generator[Any, tuple[Any, ...], Any],
    *,
    flush_partial: bool = False,
) -> Generator[None, Any, None]:
    if isinstance(size, bool) or not isinstance(size, int) or size <= 0:
        raise ValueError("size must be a positive integer")

    batch: list[Any] = []

    try:
        while True:
            item = yield
            batch.append(item)

            if len(batch) == size:
                target.send(tuple(batch))
                batch.clear()
    finally:
        if flush_partial and batch:
            target.send(tuple(batch))
        target.close()

## Step 2: test complete batches

In [28]:
complete_batches: list[tuple[int, ...]] = []
complete_sink = list_sink(complete_batches)
complete_batcher = batch_stage(3, complete_sink)

for number in range(1, 8):
    complete_batcher.send(number)

complete_batcher.close()

assert complete_batches == [(1, 2, 3), (4, 5, 6)]

## Step 3: test partial flushing

In [29]:
all_batches: list[tuple[int, ...]] = []
all_sink = list_sink(all_batches)
flushing_batcher = batch_stage(3, all_sink, flush_partial=True)

for number in range(1, 8):
    flushing_batcher.send(number)

flushing_batcher.close()

assert all_batches == [(1, 2, 3), (4, 5, 6), (7,)]

## Step 4: test configuration errors

The generator object is created first, but the validation body runs when the
decorator calls `next(...)`.

In [30]:
try:
    batch_stage(0, list_sink([]))
except ValueError as exc:
    assert "positive integer" in str(exc)
else:
    raise AssertionError("ValueError was expected")

# Problem 7 — Build a transactional buffer

## Scenario

A stream processor receives values that should not become permanent immediately.
The caller can:

- append data to a pending transaction;
- commit all pending data;
- roll back all pending data;
- finish and obtain a final summary.

We will represent the protocol with dataclass command objects sent through `.send()`.

## Step 1: define commands and responses

In [31]:
@dataclass(frozen=True)
class AddItem:
    value: Any


@dataclass(frozen=True)
class Commit:
    pass


@dataclass(frozen=True)
class Rollback:
    pass


@dataclass(frozen=True)
class Finish:
    pass


@dataclass(frozen=True)
class TransactionState:
    committed_count: int
    pending_count: int

## Step 2: decide how termination returns data

A generator can return a value with `return result`.
The caller receives that value through `StopIteration.value`.

This is different from `.close()`, which does not expose a normal return value.

In [32]:
@auto_prime
def transactional_buffer(
    committed: list[Any],
) -> Generator[TransactionState, AddItem | Commit | Rollback | Finish, tuple[Any, ...]]:
    pending: list[Any] = []
    state = TransactionState(
        committed_count=len(committed),
        pending_count=0,
    )

    while True:
        command = yield state

        if isinstance(command, AddItem):
            pending.append(command.value)

        elif isinstance(command, Commit):
            committed.extend(pending)
            pending.clear()

        elif isinstance(command, Rollback):
            pending.clear()

        elif isinstance(command, Finish):
            return tuple(committed)

        else:
            raise TypeError(f"unsupported command: {command!r}")

        state = TransactionState(
            committed_count=len(committed),
            pending_count=len(pending),
        )

## Step 3: walk through one transaction

In [33]:
permanent: list[str] = []
transaction = transactional_buffer(permanent)

assert transaction.send(AddItem("A")) == TransactionState(0, 1)
assert transaction.send(AddItem("B")) == TransactionState(0, 2)
assert transaction.send(Commit()) == TransactionState(2, 0)

assert transaction.send(AddItem("C")) == TransactionState(2, 1)
assert transaction.send(Rollback()) == TransactionState(2, 0)

assert permanent == ["A", "B"]

## Step 4: finish and capture the return value

In [34]:
try:
    transaction.send(Finish())
except StopIteration as exc:
    final_committed = exc.value
else:
    raise AssertionError("The coroutine should have terminated")

assert final_committed == ("A", "B")
assert inspect.getgeneratorstate(transaction) == "GEN_CLOSED"

## Step 5: understand protocol violations

The implementation intentionally raises `TypeError` for unknown commands.
Because the exception is unhandled inside the coroutine, the coroutine closes.
This is appropriate for a programming error in the caller.

In [35]:
invalid_transaction = transactional_buffer([])

try:
    invalid_transaction.send("COMMIT")
except TypeError as exc:
    assert "unsupported command" in str(exc)

assert inspect.getgeneratorstate(invalid_transaction) == "GEN_CLOSED"

# Problem 8 — Create a dynamic publish/subscribe router

## Goal

Build one coroutine that manages a set of subscriber coroutines.

The router accepts three command types:

- subscribe a target;
- unsubscribe a target;
- publish a message to all active targets.

A subscriber that has already closed should be removed automatically during
publishing.

## Step 1: model router commands

In [36]:
@dataclass(frozen=True)
class Subscribe:
    target: Generator[Any, Any, Any]


@dataclass(frozen=True)
class Unsubscribe:
    target: Generator[Any, Any, Any]


@dataclass(frozen=True)
class Publish:
    message: Any


@dataclass(frozen=True)
class RouterState:
    subscriber_count: int
    last_delivery_count: int

## Step 2: implement the router

A list is used instead of a set because generator objects are identity-based
resources and deterministic delivery order is useful for testing.

In [37]:
@auto_prime
def pubsub_router() -> Generator[
    RouterState,
    Subscribe | Unsubscribe | Publish,
    None,
]:
    subscribers: list[Generator[Any, Any, Any]] = []
    state = RouterState(subscriber_count=0, last_delivery_count=0)

    try:
        while True:
            command = yield state
            delivered = 0

            if isinstance(command, Subscribe):
                if command.target not in subscribers:
                    subscribers.append(command.target)

            elif isinstance(command, Unsubscribe):
                if command.target in subscribers:
                    subscribers.remove(command.target)

            elif isinstance(command, Publish):
                survivors: list[Generator[Any, Any, Any]] = []

                for subscriber in subscribers:
                    try:
                        subscriber.send(command.message)
                    except StopIteration:
                        continue
                    else:
                        delivered += 1
                        survivors.append(subscriber)

                subscribers = survivors

            else:
                raise TypeError(f"unsupported router command: {command!r}")

            state = RouterState(
                subscriber_count=len(subscribers),
                last_delivery_count=delivered,
            )
    finally:
        for subscriber in subscribers:
            subscriber.close()

## Step 3: create two subscribers

In [38]:
left_messages: list[str] = []
right_messages: list[str] = []

left_subscriber = tagged_collector(left_messages, "L")
right_subscriber = tagged_collector(right_messages, "R")

router = pubsub_router()

## Step 4: subscribe and publish

In [39]:
assert router.send(Subscribe(left_subscriber)) == RouterState(1, 0)
assert router.send(Subscribe(right_subscriber)) == RouterState(2, 0)

assert router.send(Publish("one")) == RouterState(2, 2)
assert router.send(Publish("two")) == RouterState(2, 2)

assert left_messages == ["L:one", "L:two"]
assert right_messages == ["R:one", "R:two"]

## Step 5: unsubscribe one target

In [40]:
assert router.send(Unsubscribe(right_subscriber)) == RouterState(1, 0)
assert router.send(Publish("three")) == RouterState(1, 1)

assert left_messages == ["L:one", "L:two", "L:three"]
assert right_messages == ["R:one", "R:two"]

## Step 6: close the router

The router owns all still-subscribed targets and closes them.
The unsubscribed target remains the caller's responsibility.

In [41]:
router.close()

assert inspect.getgeneratorstate(left_subscriber) == "GEN_CLOSED"
assert inspect.getgeneratorstate(right_subscriber) == "GEN_SUSPENDED"

right_subscriber.close()

# Problem 9 — Sliding-window anomaly detector

## Goal

Create a coroutine that receives numeric measurements and returns an anomaly
report.

A value is anomalous when there are already enough historical samples and its
distance from the historical mean is greater than a configurable number of
historical standard deviations.

Important design choice: the current value is compared with the **previous**
window, then appended to the window.

## Step 1: define the report

In [42]:
@dataclass(frozen=True)
class AnomalyReport:
    value: float
    historical_count: int
    historical_mean: float | None
    historical_stddev: float | None
    anomalous: bool

## Step 2: implement a helper for population standard deviation

For a small fixed window, recomputing from the deque keeps the example readable.
A very high-throughput system could maintain additional online state.

In [43]:
def population_stddev(values: Iterable[float], mean: float) -> float:
    materialized = tuple(values)

    if not materialized:
        return 0.0

    variance = sum((value - mean) ** 2 for value in materialized) / len(materialized)
    return math.sqrt(variance)

## Step 3: implement the detector

In [44]:
@auto_prime
def anomaly_detector(
    window_size: int,
    *,
    minimum_history: int = 3,
    threshold: float = 2.0,
) -> Generator[AnomalyReport | None, Any, None]:
    if window_size <= 0:
        raise ValueError("window_size must be positive")

    if minimum_history <= 0 or minimum_history > window_size:
        raise ValueError("minimum_history must be between 1 and window_size")

    if threshold <= 0:
        raise ValueError("threshold must be positive")

    history: deque[float] = deque(maxlen=window_size)
    report: AnomalyReport | None = None

    while True:
        value = yield report

        if isinstance(value, bool) or not isinstance(value, Real):
            continue

        numeric = float(value)

        if not math.isfinite(numeric):
            continue

        count = len(history)

        if count >= minimum_history:
            mean = sum(history) / count
            stddev = population_stddev(history, mean)

            if stddev == 0.0:
                anomalous = numeric != mean
            else:
                anomalous = abs(numeric - mean) > threshold * stddev
        else:
            mean = None
            stddev = None
            anomalous = False

        report = AnomalyReport(
            value=numeric,
            historical_count=count,
            historical_mean=mean,
            historical_stddev=stddev,
            anomalous=anomalous,
        )

        history.append(numeric)

## Step 4: build stable history

In [45]:
detector = anomaly_detector(window_size=5, minimum_history=3, threshold=2.0)

r1 = detector.send(10)
r2 = detector.send(10)
r3 = detector.send(10)

assert r1 is not None and r1.historical_count == 0
assert r2 is not None and r2.historical_count == 1
assert r3 is not None and r3.historical_count == 2
assert not r1.anomalous and not r2.anomalous and not r3.anomalous

## Step 5: detect a large change

The fourth value is compared with the previous three values, all equal to 10.
The historical standard deviation is zero, so any different value is anomalous.

In [46]:
r4 = detector.send(50)

assert r4 is not None
assert r4.historical_count == 3
assert r4.historical_mean == 10.0
assert r4.historical_stddev == 0.0
assert r4.anomalous is True

detector.close()

# Problem 10 — Wrap coroutine lifecycle in a context manager

## Motivation

Coroutines often own downstream stages or other resources.
Relying on callers to remember `.close()` is fragile.

We will build a context-managed handle:

```python
with CoroutineSession(my_coroutine()) as session:
    session.send(...)
```

The coroutine closes automatically when the block exits.

## Step 1: define a generic handle

In [47]:
OutputT = TypeVar("OutputT")
SentT = TypeVar("SentT")
FinalT = TypeVar("FinalT")


class CoroutineSession(
    AbstractContextManager["CoroutineSession[OutputT, SentT, FinalT]"],
    Generic[OutputT, SentT, FinalT],
):
    def __init__(self, generator: Generator[OutputT, SentT, FinalT]) -> None:
        if not inspect.isgenerator(generator):
            raise TypeError("generator must be a generator object")
        self._generator = generator

    @property
    def state(self) -> str:
        return inspect.getgeneratorstate(self._generator)

    @property
    def closed(self) -> bool:
        return self.state == "GEN_CLOSED"

    def send(self, value: SentT) -> OutputT:
        if self.closed:
            raise RuntimeError("cannot send to a closed coroutine")
        return self._generator.send(value)

    def throw(self, error: BaseException | type[BaseException]) -> OutputT:
        if self.closed:
            raise RuntimeError("cannot throw into a closed coroutine")
        return self._generator.throw(error)

    def close(self) -> None:
        self._generator.close()

    def __exit__(self, exc_type, exc, traceback) -> None:
        self.close()
        return None

## Step 2: verify automatic cleanup on normal exit

In [48]:
with CoroutineSession(controlled_total()) as session:
    assert session.state == "GEN_SUSPENDED"
    assert session.send(8) == TotalState(8.0, False)
    assert session.throw(Freeze) == TotalState(8.0, True)

assert session.closed

## Step 3: verify cleanup when the caller raises an exception

The context manager closes the coroutine but does not suppress the caller's error.

In [49]:
exception_session: CoroutineSession[TotalState, Any, None] | None = None

try:
    with CoroutineSession(controlled_total()) as exception_session:
        exception_session.send(3)
        raise LookupError("caller failure")
except LookupError:
    pass

assert exception_session is not None
assert exception_session.closed

# Problem 11 — Build a reusable coroutine test harness

## Goal

Testing coroutine code manually is repetitive.
Create a helper that:

- constructs an already-primed coroutine;
- verifies its initial state;
- sends a sequence of inputs;
- collects yielded responses;
- closes the coroutine in `finally`;
- verifies the closed state.

The harness is intentionally small, but it captures the most important lifecycle
checks.

In [50]:
def exercise_coroutine(
    factory: Callable[[], Generator[Any, Any, Any]],
    inputs: Iterable[Any],
) -> list[Any]:
    coroutine = factory()

    assert inspect.getgeneratorstate(coroutine) == "GEN_SUSPENDED"

    responses: list[Any] = []

    try:
        for item in inputs:
            responses.append(coroutine.send(item))
    finally:
        coroutine.close()

    assert inspect.getgeneratorstate(coroutine) == "GEN_CLOSED"
    return responses

## Use the harness with a new coroutine

This coroutine returns the length of the longest string seen so far.
Non-string inputs leave the state unchanged.

In [51]:
@auto_prime
def longest_length() -> Generator[int, Any, None]:
    longest = 0

    while True:
        value = yield longest
        if isinstance(value, str):
            longest = max(longest, len(value))

In [52]:
lengths = exercise_coroutine(
    longest_length,
    ["a", "abcd", 999, "xy", "abcdefgh"],
)

assert lengths == [1, 4, 4, 4, 8]

# Problem 12 — Capstone: validated sensor pipeline with diagnostics

## Scenario

A sensor sends dictionaries such as:

```python
{"sensor": "A", "reading": 21.5}
```

The system must:

1. reject non-dictionaries;
2. require a non-empty sensor name;
3. parse finite numeric readings;
4. reject booleans;
5. route invalid records to a diagnostic sink;
6. maintain the latest valid reading for every sensor;
7. emit an immutable snapshot after each accepted record;
8. close all stages when the entry coroutine closes.

We will construct the system in small pieces.

## Step 1: model diagnostics and snapshots

In [53]:
@dataclass(frozen=True)
class Diagnostic:
    record: Any
    reason: str


@dataclass(frozen=True)
class SensorSnapshot:
    latest: tuple[tuple[str, float], ...]

The snapshot stores sorted key/value pairs rather than a mutable dictionary.
This makes snapshots deterministic, immutable, and easy to compare in tests.

## Step 2: create a diagnostic sink

In [54]:
@auto_prime
def diagnostic_sink(
    diagnostics: list[Diagnostic],
) -> Generator[None, Diagnostic, None]:
    while True:
        diagnostics.append((yield))

## Step 3: create the state-owning aggregation stage

In [55]:
@auto_prime
def latest_reading_aggregator(
    snapshots: list[SensorSnapshot],
) -> Generator[None, tuple[str, float], None]:
    latest: dict[str, float] = {}

    while True:
        sensor, reading = yield
        latest[sensor] = reading

        snapshot = SensorSnapshot(
            latest=tuple(sorted(latest.items())),
        )
        snapshots.append(snapshot)

## Step 4: create the validation and parsing stage

The parser forwards valid `(sensor, reading)` pairs to the aggregator.
Invalid records are converted into diagnostics.

In [56]:
@auto_prime
def sensor_validator(
    valid_target: Generator[Any, tuple[str, float], Any],
    invalid_target: Generator[Any, Diagnostic, Any],
) -> Generator[None, Any, None]:
    try:
        while True:
            record = yield

            if not isinstance(record, dict):
                invalid_target.send(
                    Diagnostic(record, "record must be a dictionary")
                )
                continue

            sensor = record.get("sensor")
            reading = record.get("reading")

            if not isinstance(sensor, str) or not sensor.strip():
                invalid_target.send(
                    Diagnostic(record, "sensor must be a non-empty string")
                )
                continue

            if isinstance(reading, bool):
                invalid_target.send(
                    Diagnostic(record, "boolean readings are not accepted")
                )
                continue

            try:
                numeric = float(reading)
            except (TypeError, ValueError):
                invalid_target.send(
                    Diagnostic(record, "reading must be numeric")
                )
                continue

            if not math.isfinite(numeric):
                invalid_target.send(
                    Diagnostic(record, "reading must be finite")
                )
                continue

            valid_target.send((sensor.strip(), numeric))
    finally:
        valid_target.close()
        invalid_target.close()

## Step 5: assemble the stages

In [57]:
sensor_snapshots: list[SensorSnapshot] = []
sensor_diagnostics: list[Diagnostic] = []

diagnostics_stage = diagnostic_sink(sensor_diagnostics)
aggregation_stage = latest_reading_aggregator(sensor_snapshots)
sensor_entry = sensor_validator(aggregation_stage, diagnostics_stage)

## Step 6: send mixed-quality input

In [58]:
sensor_records = [
    {"sensor": "A", "reading": 21.5},
    {"sensor": "B", "reading": "18.25"},
    {"sensor": "A", "reading": 22},
    {"sensor": "", "reading": 10},
    {"sensor": "C", "reading": True},
    {"sensor": "D", "reading": "unknown"},
    {"sensor": "E", "reading": float("inf")},
    ["not", "a", "dictionary"],
]

for record in sensor_records:
    sensor_entry.send(record)

## Step 7: verify accepted snapshots

There are three valid records. Sensor `A` is updated by the third valid record.

In [59]:
assert sensor_snapshots == [
    SensorSnapshot((("A", 21.5),)),
    SensorSnapshot((("A", 21.5), ("B", 18.25))),
    SensorSnapshot((("A", 22.0), ("B", 18.25))),
]

## Step 8: verify diagnostic reasons

In [60]:
assert [item.reason for item in sensor_diagnostics] == [
    "sensor must be a non-empty string",
    "boolean readings are not accepted",
    "reading must be numeric",
    "reading must be finite",
    "record must be a dictionary",
]

## Step 9: close the entry and verify the complete lifecycle

In [61]:
sensor_entry.close()

assert inspect.getgeneratorstate(sensor_entry) == "GEN_CLOSED"
assert inspect.getgeneratorstate(aggregation_stage) == "GEN_CLOSED"
assert inspect.getgeneratorstate(diagnostics_stage) == "GEN_CLOSED"

# Concept review

## What the decorator does

The decorator removes a repetitive setup step:

```python
generator = coroutine_function(...)
next(generator)
```

and replaces it with:

```python
generator = coroutine_function(...)
```

The returned object is still a generator. The decorator only changes how it is
created and initialized.

## What the decorator does not do

It does not:

- make the code concurrent;
- run work in the background;
- create threads;
- create an event loop;
- convert the generator into an `async def` coroutine;
- automatically recover from exceptions raised after priming.

The generator remains cooperatively driven by explicit calls such as `.send()`,
`.throw()`, `.close()`, and `next()`.

## Error-handling rule of thumb

Catch errors inside a coroutine only when they are part of the intended data or
control protocol.

Good examples:

- malformed external input;
- a documented `Freeze` control signal;
- a closed downstream subscriber.

Usually do not catch:

- `NameError`;
- unexpected `AttributeError`;
- arbitrary `Exception`;
- programming errors that should fail loudly during development.

## Ownership rule of thumb

A stage should close a downstream coroutine only when it owns that coroutine.

In the pipeline examples, upstream stages own their downstream stages, so closure
propagates through `finally`.

In the publish/subscribe example, unsubscribing transfers responsibility back to
the caller; therefore, an unsubscribed coroutine is not closed by the router.

# Additional practice problems

These exercises are left for independent work. Each includes a solution direction.

## Practice A — Time-window counter

Receive monotonically increasing timestamps and return the number of timestamps in
the trailing 30 seconds.

**Suggested solution:** store timestamps in a `deque`; before appending the new
timestamp, repeatedly remove values smaller than `current - 30`.

## Practice B — Retry-aware delivery stage

A downstream target may raise a custom `TemporaryUnavailable` exception.
Retry the current item up to a configurable limit before routing it to an error
sink.

**Suggested solution:** keep retry logic around a single `target.send(item)` call;
never catch unrelated exceptions.

## Practice C — Multi-field aggregation

Receive dictionaries containing `department` and `amount`.
Maintain count, total, and mean for each department.

**Suggested solution:** store a small mutable state object per department, but emit
new immutable snapshots to callers.

## Practice D — Protocol handshake validation

Extend `prime_and_capture` so the decorator accepts a predicate for validating the
initial yielded value.

**Suggested solution:** call the predicate immediately after `next(generator)`;
close the generator before raising when validation fails.

## Practice E — Coroutine graph shutdown

Build a router in which one input is forwarded through several independent
pipelines. Ensure that closing the root closes every unique stage exactly once.

**Suggested solution:** track owned generators by identity and centralize shutdown
in one coordinator object.

# Best-practice checklist

- Prime with `next(generator)` or `generator.send(None)`.
- Preserve metadata with `@wraps`.
- Forward `*args` and `**kwargs`.
- Validate the decorated function's return type.
- Treat termination before the first `yield` as a setup error.
- Place cleanup in `finally`.
- Use narrow exception handling.
- Represent nontrivial protocols with named command objects.
- Distinguish `.close()` from normal `return`.
- Test generator states when lifecycle correctness matters.
- Decide explicitly which component owns and closes each coroutine.
- Prefer `async def` and `asyncio` for modern asynchronous I/O.

# Final verification

Running the notebook from top to bottom should complete without assertion failures.

In [62]:
print("All tutorial-style advanced coroutine problems passed.")

All tutorial-style advanced coroutine problems passed.
